In [1]:
import asyncio
import json
import re
from dataclasses import asdict, dataclass
from pathlib import Path
from urllib.parse import urljoin

import httpx
import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from tenacity import retry, stop_after_attempt, wait_exponential

nest_asyncio.apply()

DEFAULT_URL = "https://canadabuys.canada.ca/en/tender-opportunities/tender-notice/ws4286933967-doc4822970058"
BASE_URL = "https://canadabuys.canada.ca"


@dataclass
class CompanyRecord:
    tender_url: str
    tender_title: str | None = None
    tender_solicitation_number: str | None = None
    tender_publication_date: str | None = None
    tender_closing_datetime: str | None = None

    company_name: str | None = None
    partner_page_url: str | None = None
    company_website: str | None = None
    company_tagline: str | None = None
    company_description: str | None = None

    contact_first_name: str | None = None
    contact_last_name: str | None = None
    contact_full_name: str | None = None
    contact_title: str | None = None
    contact_phone: str | None = None
    contact_email: str | None = None

    other_links_json: str | None = None
    date_modified: str | None = None

    scraped_ok: bool = False
    scrape_error: str | None = None


def clean_text(text):
    if text is None:
        return None
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text or None


def safe_json_dumps(obj):
    if obj in (None, {}, [], ""):
        return None
    return json.dumps(obj, ensure_ascii=False)


def extract_email(text):
    m = re.search(r'[\w.+-]+@[\w.-]+\.\w+', text)
    return m.group(0) if m else None


def extract_phone(text):
    patterns = [
        r'Telephone\s*[:\-]?\s*(\(?\d{3}\)?[\s\-]?\d{3}[\s\-]?\d{4})',
        r'(\(?\d{3}\)?[\s\-]?\d{3}[\s\-]?\d{4})',
    ]
    for p in patterns:
        m = re.search(p, text, flags=re.IGNORECASE)
        if m:
            return clean_text(m.group(1))
    return None


def get_first_anchor_href_near_text(soup, text_snippet):
    snippet = soup.find(string=lambda s: isinstance(s, str) and text_snippet.lower() in s.lower())
    if not snippet:
        return None
    parent = snippet.parent
    if not parent:
        return None
    a = parent.find("a", href=True)
    if a:
        return urljoin(BASE_URL, a["href"])
    nxt = parent.find_next("a", href=True)
    if nxt:
        return urljoin(BASE_URL, nxt["href"])
    return None


def parse_tender_metadata(soup, tender_url):
    text = soup.get_text("\n", strip=True)

    title = None
    h1 = soup.find("h1")
    if h1:
        title = clean_text(h1.get_text(" ", strip=True))

    solicitation_number = None
    m = re.search(r"Solicitation number\s+([A-Z0-9\-]+)", text, flags=re.IGNORECASE)
    if m:
        solicitation_number = m.group(1).strip()

    publication_date = None
    m = re.search(r"Publication date\s+([0-9]{4}/[0-9]{2}/[0-9]{2})", text)
    if m:
        publication_date = m.group(1)

    closing_dt = None
    m = re.search(r"Closing date and time\s+([0-9]{4}/[0-9]{2}/[0-9]{2}\s+[0-9]{2}:[0-9]{2}\s+[A-Z]{2,4})", text)
    if m:
        closing_dt = m.group(1)

    return {
        "tender_url": tender_url,
        "tender_title": title,
        "tender_solicitation_number": solicitation_number,
        "tender_publication_date": publication_date,
        "tender_closing_datetime": closing_dt,
    }


def parse_partner_links(soup):
    partners = []

    for a in soup.select("a[href]"):
        href = a.get("href", "")
        name = clean_text(a.get_text(" ", strip=True))
        if not name:
            continue

        full_url = urljoin(BASE_URL, href)

        if "/node/preview/" in full_url:
            partners.append((name, full_url))

    seen = set()
    unique = []
    for name, url in partners:
        key = (name.lower(), url)
        if key not in seen:
            seen.add(key)
            unique.append((name, url))

    return unique


def parse_company_page(html, page_url, tender_meta):
    soup = BeautifulSoup(html, "lxml")
    page_text = soup.get_text("\n", strip=True)

    company_name = None
    h1 = soup.find("h1")
    if h1:
        company_name = clean_text(h1.get_text(" ", strip=True))

    company_website = get_first_anchor_href_near_text(soup, "Company website")

    company_tagline = None
    company_description = None

    desc_match = re.search(
        r"Company website.*?\n(.+?)\nFirst name\s+",
        page_text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if desc_match:
        parts = [p.strip() for p in re.split(r"\n+", desc_match.group(1).strip()) if p.strip()]
        if parts:
            if parts[0].startswith("http"):
                parts = parts[1:]
            if parts:
                company_tagline = clean_text(parts[0])
                company_description = clean_text(" ".join(parts[1:]) if len(parts) > 1 else parts[0])

    if not company_description:
        lines = [clean_text(x) for x in page_text.splitlines()]
        lines = [x for x in lines if x]
        try:
            start_idx = next(i for i, x in enumerate(lines) if x.startswith("Company website"))
            end_idx = next(i for i, x in enumerate(lines) if x == "First name")
            block = lines[start_idx + 1:end_idx]
            if block and block[0].startswith("http"):
                block = block[1:]
            if block:
                company_tagline = block[0]
                company_description = clean_text(" ".join(block[1:]) if len(block) > 1 else block[0])
        except StopIteration:
            pass

    first_name = None
    last_name = None
    contact_title = None
    date_modified = None

    m = re.search(r"First name\s+([^\n]+)", page_text, flags=re.IGNORECASE)
    if m:
        first_name = clean_text(m.group(1))

    m = re.search(r"Last name\s+([^\n]+)", page_text, flags=re.IGNORECASE)
    if m:
        last_name = clean_text(m.group(1))

    m = re.search(r"Title/position\s+([^\n]+)", page_text, flags=re.IGNORECASE)
    if m:
        contact_title = clean_text(m.group(1))

    m = re.search(r"Date modified:\s+([0-9]{4}-[0-9]{2}-[0-9]{2})", page_text, flags=re.IGNORECASE)
    if m:
        date_modified = m.group(1)

    contact_email = extract_email(page_text)
    contact_phone = extract_phone(page_text)
    contact_full_name = clean_text(" ".join(x for x in [first_name, last_name] if x))

    other_links = []
    for a in soup.select("a[href]"):
        href = urljoin(BASE_URL, a["href"])
        label = clean_text(a.get_text(" ", strip=True))
        if not href or not label:
            continue
        if href == page_url:
            continue
        if "linkedin.com" in href.lower():
            other_links.append({"label": label, "url": href})
        elif company_website and href != company_website and "canadabuys.canada.ca" not in href:
            other_links.append({"label": label, "url": href})

    seen_links = set()
    deduped_links = []
    for item in other_links:
        key = (item["label"], item["url"])
        if key not in seen_links:
            seen_links.add(key)
            deduped_links.append(item)

    return CompanyRecord(
        **tender_meta,
        company_name=company_name,
        partner_page_url=page_url,
        company_website=company_website,
        company_tagline=company_tagline,
        company_description=company_description,
        contact_first_name=first_name,
        contact_last_name=last_name,
        contact_full_name=contact_full_name,
        contact_title=contact_title,
        contact_phone=contact_phone,
        contact_email=contact_email,
        other_links_json=safe_json_dumps(deduped_links),
        date_modified=date_modified,
        scraped_ok=True,
    )


@retry(wait=wait_exponential(multiplier=1, min=1, max=10), stop=stop_after_attempt(4))
async def fetch_text(client, url):
    resp = await client.get(url, follow_redirects=True, timeout=45.0)
    resp.raise_for_status()
    return resp.text


async def scrape_company(client, semaphore, company_name, page_url, tender_meta):
    async with semaphore:
        try:
            html = await fetch_text(client, page_url)
            return parse_company_page(html, page_url, tender_meta)
        except Exception as e:
            return CompanyRecord(
                **tender_meta,
                company_name=company_name,
                partner_page_url=page_url,
                scraped_ok=False,
                scrape_error=f"{type(e).__name__}: {e}",
            )


async def scrape_all_notebook(
        tender_url=DEFAULT_URL,
        concurrency=10,
        save_excel=True,
        output_dir="output"
):
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; CanadaBuysScraper/1.0)"
    }

    limits = httpx.Limits(max_connections=20, max_keepalive_connections=10)

    async with httpx.AsyncClient(headers=headers, limits=limits) as client:
        tender_html = await fetch_text(client, tender_url)
        tender_soup = BeautifulSoup(tender_html, "lxml")

        tender_meta = parse_tender_metadata(tender_soup, tender_url)
        partner_links = parse_partner_links(tender_soup)

        if not partner_links:
            raise RuntimeError("No partner company links found on the tender page.")

        print(f"Found {len(partner_links)} partner links")

        semaphore = asyncio.Semaphore(concurrency)

        tasks = [
            scrape_company(client, semaphore, name, url, tender_meta)
            for name, url in partner_links
        ]

        results = await asyncio.gather(*tasks)

    df = pd.DataFrame([asdict(r) for r in results])

    if not df.empty:
        df["company_name_normalized"] = (
            df["company_name"]
            .fillna("")
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    if save_excel:
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        csv_file = output_path / "canadabuys_ai_source_list_partners.csv"
        xlsx_file = output_path / "canadabuys_ai_source_list_partners.xlsx"
        json_file = output_path / "canadabuys_ai_source_list_partners.json"

        df.to_csv(csv_file, index=False)
        df.to_json(json_file, orient="records", force_ascii=False, indent=2)

        with pd.ExcelWriter(xlsx_file, engine="openpyxl") as writer:
            df.to_excel(writer, sheet_name="companies", index=False)

            deduped = (
                df.sort_values(["company_name", "partner_page_url"])
                .drop_duplicates(subset=["company_name_normalized"], keep="first")
            )
            deduped.to_excel(writer, sheet_name="deduped_companies", index=False)

            failures = df.loc[~df["scraped_ok"].fillna(False)].copy()
            failures.to_excel(writer, sheet_name="failures", index=False)

        print(f"Saved: {csv_file}")
        print(f"Saved: {xlsx_file}")
        print(f"Saved: {json_file}")

    return df

In [2]:
!pip install httpx[http2] beautifulsoup4 lxml pandas openpyxl tenacity tqdm

zsh:1: no matches found: httpx[http2]


In [3]:
import sys
!{sys.executable} -m pip install lxml

In [7]:
df = await scrape_all_notebook(
    tender_url="https://canadabuys.canada.ca/en/tender-opportunities/tender-notice/ws4286933967-doc4822970058",
    concurrency=10,
    save_excel=True,
    output_dir="output"
)

df.head(20)

Found 159 partner links
Saved: output/canadabuys_ai_source_list_partners.csv
Saved: output/canadabuys_ai_source_list_partners.xlsx
Saved: output/canadabuys_ai_source_list_partners.json


,tender_url,tender_title,tender_solicitation_number,tender_publication_date,tender_closing_datetime,company_name,partner_page_url,company_website,company_tagline,company_description,...,contact_last_name,contact_full_name,contact_title,contact_phone,contact_email,other_links_json,date_modified,scraped_ok,scrape_error,company_name_normalized
0,https://canadabuys.canada.ca/en/tender-opportu...,Invitation to Qualify to Artificial Intelligen...,WS4286933967,2024/10/08,2026/09/30\n14:00 EDT,Web Inventix AI / FranOps (Franchise Operating...,https://canadabuys.canada.ca/en/node/preview/1...,https://webinventix.ai,"AI automation, APIs, data systems, platform in...",Web Inventix AI and FranOps deliver production...,...,Gratton,Dan Gratton,Founder | Strategic Advisor,(519) 770 8331,dan@webinventix.ai,"[{""label"": ""Jobs and the workplace"", ""url"": ""h...",None,True,None,web inventix ai / franops (franchise operating...
1,https://canadabuys.canada.ca/en/tender-opportu...,Invitation to Qualify to Artificial Intelligen...,WS4286933967,2024/10/08,2026/09/30\n14:00 EDT,The Altercation Company,https://canadabuys.canada.ca/en/node/preview/1...,https://augureai.ca/,"The most powerful, fully sovereign AI models i...",Augure builds Canada's most powerful fully sov...,...,Dooley,Nicholas Dooley,Founder,(647) 293 7581,nick@altercation.io,"[{""label"": ""Jobs and the workplace"", ""url"": ""h...",None,True,None,the altercation company
2,https://canadabuys.canada.ca/en/tender-opportu...,Invitation to Qualify to Artificial Intelligen...,WS4286933967,2024/10/08,2026/09/30\n14:00 EDT,LearningBrick,https://canadabuys.canada.ca/en/node/preview/1...,https://www.learningbrick.com,LearningBrick provides comprehensive AI traini...,LearningBrick is a forward-thinking IT and tec...,...,Lennox,Bill Lennox,Vice President of Sales and Operations,(613) 802 9453,Bill@learningbrick.com,"[{""label"": ""Jobs and the workplace"", ""url"": ""h...",None,True,None,learningbrick
3,https://canadabuys.canada.ca/en/tender-opportu...,Invitation to Qualify to Artificial Intelligen...,WS4286933967,2024/10/08,2026/09/30\n14:00 EDT,Northern Alberta Institute of Technology,https://canadabuys.canada.ca/en/node/preview/1...,https://www.nait.ca/applied-research/home,AI - Machine Learning - Generative AI - Agenti...,Most organizations hit the same wall when adop...,...,Mirzaei,Maryam Mirzaei,Applied Research Chair,(780) 680 6532,maryamm@nait.ca,"[{""label"": ""Jobs and the workplace"", ""url"": ""h...",None,True,None,northern alberta institute of technology
4,https://canadabuys.canada.ca/en/tender-opportu...,Invitation to Qualify to Artificial Intelligen...,WS4286933967,2024/10/08,2026/09/30\n14:00 EDT,Graf Consulting,https://canadabuys.canada.ca/en/node/preview/1...,http://www.grafai.agency,"AI automation, digital transformation & busine...",Graf AI Agency helps businesses modernize how ...,...,Nwokeabia,Nnenna Nwokeabia,Owner,(902) 329 7576,info@grafconsultings.com,"[{""label"": ""Jobs and the workplace"", ""url"": ""h...",None,True,None,graf consulting
5,https://canadabuys.canada.ca/en/tender-opportu...,Invitation to Qualify to Artificial Intelligen...,WS4286933967,2024/10/08,2026/09/30\n14:00 EDT,SAI Systems,https://canadabuys.canada.ca/en/node/preview/1...,https://saicloudsecure.com/,AI Architect and Security Security Assessor IS...,AI Architect and Security Security Assessor IS...,...,Patterson,Keith Patterson,CTO,(613) 218 3057,KP@saicloudsecure.com,"[{""label"": ""Jobs and the workplace"", ""url"": ""h...",None,True,None,sai systems
6,https://canadabuys.canada.ca/en/tender-opportu...,Invitation to Qualify to Artificial Intelligen...,WS4286933967,2024/10/08,2026/09/30\n14:00 EDT,AMG SOLUTIONS INC.,https://canadabuys.canada.ca/en/node/preview/1...,http://www.amgsolutions.one,Implantation de systèmes IA pour croissance d’...,AMG Solutions Inc. est une entreprise spéciali...,...,Mauricin,Patrick Mauricin,Fondateur,(548) 328 4410,pj.mauricin@gmail.com,"[{""label"": ""Jobs and the 

In [6]:
import sys
!{sys.executable} -m pip install openpyxl